# 19 — Decisión final (futura ADR-006)

> **Estado: ESQUELETO.** Este notebook fija el diseño antes de implementar nada. Todas las secciones de trabajo son celdas `TODO` controladas por el flag `IMPLEMENTADO` (por defecto `False`): con `False` el notebook se ejecuta de principio a fin sin error y solo informa de lo pendiente; con `True` cada `TODO` sin implementar levanta `NotImplementedError`. Ninguna cifra de este texto está escrita a mano: las que aparezcan saldrán de celdas.

## 0. Objetivo

Cerrar la Fase 1 del TFM (régimen de mercado, [ADR-005](../docs/decisions/ADR-005-reencuadre-tfm-multiagente.md)).
Consolidar la evidencia del benchmark (`04`–`12`), de la fusión (`13`–`14`) y de los sintéticos
(`15`–`18`) en **una decisión documentada (futura ADR-006)** con dos resultados:

1. **Congelar la regla** que se someterá al pseudolive (`20`) antes de ver ningún dato de ese bloque.
2. **Definir el contrato de la tool `get_regimen(fecha)`**: los estados los fija el detector final
   (hipótesis de trabajo: calma / alerta / crisis ≈ normal / vigilancia / confirmado de
   `regimenes.fusion`), más la probabilidad y los días en el estado, usando solo información publicada
   hasta `fecha`.

## Preguntas que responde

1. ¿Qué detector o pareja alerta + confirmación se adopta como regla final, y con qué parámetros?
2. ¿Se adopta el aumento con sintéticos (y con qué generador/proporción) o se descarta?
3. ¿Qué afirmaciones sobre la señal de régimen quedan respaldadas y con qué incertidumbre?
4. ¿Qué estados, probabilidad y días en el estado expone `get_regimen(fecha)`?
5. ¿Qué queda explícitamente fuera (decisiones abiertas y limitaciones)?

## Entradas y salidas (vía `regimenes.rutas`)

| papel | ruta |
|---|---|
| ranking y métricas del benchmark | `rutas.RESULTS_BENCHMARK` |
| fusión D07+D08 y D02+D06 | `rutas.RESULTS_FUSION_D07_D08`, `rutas.RESULTS_FUSION_D02_D06` |
| laboratorio y aumento | `rutas.RESULTS_SINTETICOS` |
| regla congelada (salida) | `rutas.RESULTS_PSEUDOLIVE / 'regla_congelada.json'` |
| decisión (salida) | `rutas.DOCS / 'decisions' / 'ADR-006-...md'` |

## Metodología prevista

- **Evidencia consolidada**: tablas cargadas de los CSV versionados (nunca cifras a mano),
  verificando la huella de caché del benchmark (`regimenes.benchmark.load_metrics`).
- **Matriz de criterios** fijada antes de mirar el pseudolive: detección ADR-003 en ambas pistas,
  estabilidad de etiquetas, anticipación frente al azar, robustez en el control por mitades,
  coste y complejidad; ganancia del aumento solo si supera su IC en ambas pistas.
- **Congelación**: la regla (detectores, features, parámetros, umbrales de fusión, régimen de
  crisis, calendario de re-ajuste) se serializa en JSON con la huella del código
  (`regimenes.benchmark.cache`) y la fecha; desde ese momento no se modifica.
- **ADR-006**: contexto, opciones, decisión, consecuencias y criterio de éxito/fracaso del
  pseudolive, redactado desde las celdas de este notebook.

## Riesgos

- **Selección post-hoc.** Todas las opciones se han visto ya sobre el OOS: la decisión lleva sesgo
  de selección. Se mitiga pre-registrando la regla y su criterio de éxito antes de `20`
  (`lopezdeprado2018`).
- **Circularidad con sintéticos.** Si el aumento se elige con generadores condicionados a un
  detector, su ventaja puede ser artificial (ver `17`).
- **Fuga.** Ningún dato posterior a la ventana del benchmark entra en esta decisión.

**Índice**

0. Objetivo, preguntas, entradas/salidas, metodología y riesgos
1. Configuración
2. Evidencia consolidada
3. Matriz de criterios
4. Regla congelada
5. Borrador de ADR-006


## 1. Configuración

In [1]:
# Configuración común del esqueleto (no lee results/benchmark ni ajusta nada).
import yaml
import pandas as pd
from IPython.display import display

from regimenes import rutas
from regimenes import sinteticos
from regimenes.sinteticos import Generador
from regimenes.sinteticos import registry as registro_sinteticos

IMPLEMENTADO = False  # pasar a True solo cuando las secciones TODO estén implementadas

CONFIG_SINTETICOS = yaml.safe_load(rutas.SINTETICOS_CONFIG.read_text(encoding='utf-8'))
SEMILLA = CONFIG_SINTETICOS.get('semilla')


def saltar(seccion: str, detalle: str) -> None:
    """Punto TODO controlado: informa y sigue (solo se llama con IMPLEMENTADO=False)."""
    print(f'[TODO pendiente] {seccion}: {detalle}')


def tabla_rutas(rutas_nb: dict) -> pd.DataFrame:
    """Entradas/salidas del notebook (siempre vía regimenes.rutas) y si existen ya en disco."""
    filas = [
        {'papel': papel, 'ruta': ruta.relative_to(rutas.ROOT).as_posix(), 'existe': ruta.exists()}
        for papel, ruta in rutas_nb.items()
    ]
    return pd.DataFrame(filas).set_index('papel')

RUTAS_NB = {
    'entrada: benchmark': rutas.RESULTS_BENCHMARK,
    'entrada: fusión D07+D08': rutas.RESULTS_FUSION_D07_D08,
    'entrada: fusión D02+D06': rutas.RESULTS_FUSION_D02_D06,
    'entrada: sintéticos': rutas.RESULTS_SINTETICOS,
    'salida: regla congelada': rutas.RESULTS_PSEUDOLIVE / 'regla_congelada.json',
    'salida: decisiones': rutas.DOCS / 'decisions',
}
display(tabla_rutas(RUTAS_NB))


,ruta,existe
papel,,
entrada: benchmark,results/benchmark,True
entrada: fusión D07+D08,results/fusion/d07_d08,True
entrada: fusión D02+D06,results/fusion/d02_d06,True
entrada: sintéticos,results/sinteticos,True
salida: regla congelada,results/pseudolive/regla_congelada.json,False
salida: decisiones,docs/decisions,True


## 2. Evidencia consolidada

Se leerá solo la caché vigente del benchmark (si no está vigente, fallo con el comando `python -m regimenes.benchmark --track A B` a ejecutar) y los CSV de `results/fusion` y `results/sinteticos`.

In [2]:
# TODO 19.2 — evidencia consolidada
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name
from regimenes.benchmark import load_metrics, metrics_provenance

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. regimenes.benchmark.load_metrics(...) con verificación de huella; ranking ADR-003
    # 2. veredictos de fusión (verdict.csv / pair_verdict.csv) y resultados de 17-18
    raise NotImplementedError('TODO 19.2: consolidar la evidencia de 04-18')
else:
    saltar('19.2', 'evidencia consolidada')


[TODO pendiente] 19.2: evidencia consolidada


## 3. Matriz de criterios

In [3]:
# TODO 19.3 — matriz de criterios
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. opciones × criterios (fijados antes de 20) con cifras calculadas en celda
    raise NotImplementedError('TODO 19.3: construir la matriz de criterios')
else:
    saltar('19.3', 'matriz de criterios')


[TODO pendiente] 19.3: matriz de criterios


## 4. Regla congelada

In [4]:
# TODO 19.4 — serializar la regla congelada
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name
from regimenes.fusion import FusionConfig

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. dict con detectores, features, parámetros, FusionConfig, calendario de re-ajuste
    # 2. huella de código (regimenes.benchmark.cache) + fecha; escribir rutas.RESULTS_PSEUDOLIVE / 'regla_congelada.json'
    raise NotImplementedError('TODO 19.4: congelar la regla final')
else:
    saltar('19.4', 'serializar la regla congelada')


[TODO pendiente] 19.4: serializar la regla congelada


## 5. Borrador de ADR-006

In [5]:
# TODO 19.5 — borrador de ADR-006
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. generar el texto del ADR (contexto, opciones, decisión, consecuencias, criterio de éxito) desde las celdas
    raise NotImplementedError('TODO 19.5: redactar ADR-006 desde las celdas')
else:
    saltar('19.5', 'borrador de ADR-006')


[TODO pendiente] 19.5: borrador de ADR-006
